# Lab 3 - Statistical Language Modeling and Sequence Analysis
## CSET346 Natural Language Processing

## Setup

In [1]:
import nltk
nltk.download('brown')
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('universal_tagset')
nltk.download('stopwords')

from nltk.corpus import brown, gutenberg, stopwords
from nltk.util import ngrams
from nltk import FreqDist
from collections import Counter
import pandas as pd
import numpy as np
import random
import math
import re

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Q1: Corpus Analysis and N-gram Statistics (Brown Corpus - news)

In [2]:
news_words = brown.words(categories='news')
tokens = [w.lower() for w in news_words if w.isalpha()][:5000]

total_tokens = len(tokens)
vocab = set(tokens)
vocab_size = len(vocab)
print('Total tokens:', total_tokens)
print('Vocabulary size:', vocab_size)

Total tokens: 5000
Vocabulary size: 1511


In [3]:
unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

uni_fd = FreqDist(unigrams)
bi_fd = FreqDist(bigrams)
tri_fd = FreqDist(trigrams)

print('Top 10 Unigrams:')
for w, c in uni_fd.most_common(10):
    print(w, c)

print('\nTop 10 Bigrams:')
for w, c in bi_fd.most_common(10):
    print(w, c)

print('\nTop 10 Trigrams:')
for w, c in tri_fd.most_common(10):
    print(w, c)

Top 10 Unigrams:
('the',) 388
('of',) 204
('to',) 149
('a',) 129
('in',) 102
('and',) 97
('for',) 63
('that',) 51
('would',) 50
('said',) 46

Top 10 Bigrams:
('of', 'the') 64
('in', 'the') 42
('on', 'the') 16
('the', 'state') 15
('the', 'jury') 14
('for', 'the') 14
('would', 'be') 14
('to', 'the') 13
('that', 'the') 11
('at', 'the') 10

Top 10 Trigrams:
('the', 'jury', 'said') 7
('of', 'the', 'ward') 6
('the', 'grand', 'jury') 4
('is', 'expected', 'to') 4
('some', 'of', 'the') 4
('the', 'precinct', 'of') 4
('precinct', 'of', 'the') 4
('up', 'to', 'days') 4
('the', 'fulton', 'county') 3
('jury', 'said', 'it') 3


**(f) Role of a language corpus:** A corpus provides real-world text samples from which word frequencies, co-occurrence patterns, and n-gram probabilities are estimated. These statistics form the foundation of statistical language models used for prediction, tagging, and generation tasks.

## Q2: Bigram Language Model with Add-One Smoothing (Gutenberg Corpus)

In [4]:
text = gutenberg.raw('austen-emma.txt')
words = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]

unigram_counts = Counter(words)
bigram_counts = Counter(ngrams(words, 2))
V = len(set(words))

print('Vocabulary size:', V)
print('Unique bigrams:', len(bigram_counts))

Vocabulary size: 6932
Unique bigrams: 65062


In [5]:
def bigram_prob(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

def sentence_score(sentence):
    toks = [w.lower() for w in nltk.word_tokenize(sentence) if w.isalpha()]
    score = 0.0
    for w1, w2 in zip(toks, toks[1:]):
        score += math.log(bigram_prob(w1, w2))
    return score

test_sentences = [
    'Emma was very happy to see her friend.',
    'She went to the house with great pleasure.',
    'The weather was extremely pleasant that morning.'
]

for s in test_sentences:
    print(s, '->', sentence_score(s))

Emma was very happy to see her friend. -> -37.02193270008751
She went to the house with great pleasure. -> -42.78394211688338
The weather was extremely pleasant that morning. -> -46.6994077591155


**(f) Importance of smoothing:** Add-one (Laplace) smoothing prevents zero probabilities for unseen bigrams by assigning them a small non-zero probability. Without smoothing, any sentence containing an unseen bigram would receive a probability of zero, making the model unusable for real text.

## Q3: Hidden Markov Model for POS Tagging (Brown Corpus, universal tagset)

In [6]:
tagged_sents = brown.tagged_sents(categories='news', tagset='universal')
tagged_sents = list(tagged_sents)
random.seed(42)
random.shuffle(tagged_sents)

split = int(0.8 * len(tagged_sents))
train_data = tagged_sents[:split]
test_data = tagged_sents[split:]

print('Training sentences:', len(train_data))
print('Testing sentences:', len(test_data))

Training sentences: 3698
Testing sentences: 925


In [7]:
from nltk.tag import hmm

trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train_supervised(train_data)

In [8]:
for sent in test_data[:5]:
    words_only = [w for w, t in sent]
    predicted = hmm_tagger.tag(words_only)
    print(predicted)

/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])


[('In', 'ADP'), ('other', 'ADJ'), ('words', 'NOUN'), (',', '.'), ('I', 'PRON'), ('am', 'VERB'), ('getting', 'VERB'), ('only', 'ADV'), ('half', 'PRT'), ('the', 'DET'), ('dividends', 'NOUN'), ('I', 'PRON'), ('should', 'VERB'), ('.', '.')]
[('The', 'DET'), ('victim', 'NOUN'), ('Darnell', 'NOUN'), ('Somerville', 'NOUN'), (',', 'NOUN'), ('Negro', 'NOUN'), (',', 'NOUN'), ('1', 'NOUN'), (',', 'NOUN'), ('was', 'NOUN'), ('pronounced', 'NOUN'), ('dead', 'NOUN'), ('on', 'NOUN'), ('arrival', 'NOUN'), ('at', 'NOUN'), ('Anne', 'NOUN'), ('Arundel', 'NOUN'), ('General', 'NOUN'), ('Hospital', 'NOUN'), ('in', 'NOUN'), ('Annapolis', 'NOUN'), ('.', 'NOUN')]
[('Earlier', 'NOUN'), (',', 'NOUN'), ('Mitchell', 'NOUN'), ('said', 'NOUN'), ('in', 'NOUN'), ('a', 'NOUN'), ('statement', 'NOUN'), (':', 'NOUN')]
[("We've", 'PRT'), ('been', 'VERB'), ('working', 'VERB'), ('for', 'ADP'), ('weeks', 'NOUN'), ('.', '.')]
[('``', '.'), ("He'll", 'NOUN'), ('be', 'NOUN'), ('out', 'NOUN'), ('of', 'NOUN'), ('action', 'NOUN'), (

/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


In [9]:
accuracy = hmm_tagger.evaluate(test_data)
print('Test Accuracy:', accuracy)

/tmp/ipykernel_4954/65660798.py:1: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  accuracy = hmm_tagger.evaluate(test_data)


Test Accuracy: 0.5819440234414469


**(f) HMM components:**
- **Hidden states:** POS tags (e.g., NOUN, VERB), not directly observed.
- **Observations:** The actual words in the sentence.
- **Transition probabilities:** P(tag_i | tag_i-1), likelihood of one tag following another.
- **Emission probabilities:** P(word | tag), likelihood of a word given its tag.

## Q4: Ambiguity, Coreference, and Domain-Specific Language

In [10]:
ambiguity_examples = [
    ('I went to the bank to deposit money.', 'bank = financial institution'),
    ('We sat by the bank of the river.', 'bank = riverside land'),
    ('The bat flew out of the cave at night.', 'bat = flying mammal'),
    ('He hit the ball with a bat.', 'bat = sports equipment'),
    ('Please turn on the light in the room.', 'light = illumination'),
    ('This bag is very light to carry.', 'light = not heavy'),
    ('They watched the football match yesterday.', 'match = sports event'),
    ('He used a match to light the candle.', 'match = small stick for fire')
]

for sent, meaning in ambiguity_examples:
    print(sent, '->', meaning)

I went to the bank to deposit money. -> bank = financial institution
We sat by the bank of the river. -> bank = riverside land
The bat flew out of the cave at night. -> bat = flying mammal
He hit the ball with a bat. -> bat = sports equipment
Please turn on the light in the room. -> light = illumination
This bag is very light to carry. -> light = not heavy
They watched the football match yesterday. -> match = sports event
He used a match to light the candle. -> match = small stick for fire


In [11]:
coreference_examples = [
    ('Ravi went to the market because he needed vegetables.', 'he -> Ravi'),
    ('The teacher praised the students because they did well.', 'they -> the students'),
    ('Priya lost her book, but she found it later.', 'she -> Priya, it -> her book'),
    ('The dog chased the cat until it got tired.', 'it -> the cat (ambiguous, could be dog)'),
    ('John told Mike that he would help him.', 'he -> John, him -> Mike (ambiguous)')
]

for sent, ref in coreference_examples:
    print(sent, '->', ref)

Ravi went to the market because he needed vegetables. -> he -> Ravi
The teacher praised the students because they did well. -> they -> the students
Priya lost her book, but she found it later. -> she -> Priya, it -> her book
The dog chased the cat until it got tired. -> it -> the cat (ambiguous, could be dog)
John told Mike that he would help him. -> he -> John, him -> Mike (ambiguous)


In [12]:
from sklearn.datasets import fetch_20newsgroups

categories = ['sci.space', 'comp.graphics']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

text_all = ' '.join(newsgroups.data).lower()
words_ng = re.findall(r'[a-z]+', text_all)
stop_words = set(stopwords.words('english'))
filtered_words = [w for w in words_ng if w not in stop_words and len(w) > 3]

domain_fd = FreqDist(filtered_words)
print('Top 20 domain-specific terms:')
for w, c in domain_fd.most_common(20):
    print(w, c)

Top 20 domain-specific terms:
space 1054
would 591
image 540
also 468
data 435
graphics 416
nasa 414
like 389
program 355
system 318
time 313
available 310
software 294
know 289
images 285
file 279
jpeg 274
launch 270
information 257
could 257


**(f) Impact on NLP performance:** Ambiguity requires context-based disambiguation, since a single word can map to multiple meanings depending on usage. Coreference resolution is needed to correctly link pronouns to their antecedents for accurate understanding. Domain-specific vocabulary means models trained on one domain often underperform on another unless adapted, since term distributions and meanings shift across domains (e.g., 'graphics' in comp.graphics vs. general usage).